In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

def crawl_quanta_monthly_sales(start_year=2006, end_year=2026):
    """
    광달(Quanta Computer) 월간 매출 데이터 크롤링

    Parameters:
    start_year: 시작 연도
    end_year: 종료 연도

    Returns:
    DataFrame: 월간 매출 데이터
    """

    url = "https://www.quantatw.com/quanta/english/investment/financials_ms.aspx"

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        response.encoding = 'utf-8'

        soup = BeautifulSoup(response.text, 'html.parser')

        all_data = []

        for year in range(start_year, end_year + 1):
            year_div = soup.find('div', {'id': str(year), 'class': 'income'})

            if not year_div:
                print(f"연도 {year} 데이터를 찾을 수 없습니다.")
                continue

            table = year_div.find('table', {'class': 'qr'})

            if not table:
                print(f"연도 {year} 테이블을 찾을 수 없습니다.")
                continue

            rows = table.find_all('tr')

            header_found = False
            current_col_name = None
            prev_col_name = None

            for row in rows:
                cells = row.find_all(['th', 'td'])

                if not cells:
                    continue

                # 헤더 행 찾기
                if cells[0].name == 'th' and 'Month' in cells[0].get_text(strip=True):
                    header_found = True

                    # 컬럼명 추출
                    if len(cells) >= 3:
                        current_col_name = cells[1].get_text(strip=True).split('\n')[0].strip()
                        prev_col_name = cells[2].get_text(strip=True).split('\n')[0].strip()

                    continue

                # 데이터 행 처리
                if header_found and cells[0].name == 'td':
                    month_text = cells[0].get_text(strip=True)

                    # Total 행은 건너뛰기
                    if month_text == 'Total':
                        continue

                    # 월 이름을 숫자로 변환
                    month_mapping = {
                        'January': 1, 'February': 2, 'March': 3, 'April': 4,
                        'May': 5, 'June': 6, 'July': 7, 'August': 8,
                        'September': 9, 'October': 10, 'November': 11, 'December': 12
                    }

                    month_num = month_mapping.get(month_text)

                    if not month_num:
                        continue

                    # 데이터 추출
                    try:
                        current_sales = cells[1].get_text(strip=True).replace(',', '') if len(cells) > 1 else ''
                        prev_sales = cells[2].get_text(strip=True).replace(',', '') if len(cells) > 2 else ''
                        mom = cells[3].get_text(strip=True).replace('%', '') if len(cells) > 3 else ''
                        yoy = cells[4].get_text(strip=True).replace('%', '') if len(cells) > 4 else ''
                        ytd_yoy = cells[5].get_text(strip=True).replace('%', '') if len(cells) > 5 else ''

                        # 빈 값 처리
                        if current_sales and current_sales != '':
                            data_dict = {
                                'year': year,
                                'month': month_num,
                                'date': f"{year}-{month_num:02d}",
                                'current_year': current_col_name,
                                'current_sales': float(current_sales) if current_sales else None,
                                'prev_year': prev_col_name,
                                'prev_sales': float(prev_sales) if prev_sales else None,
                                'mom': float(mom) if mom else None,
                                'yoy': float(yoy) if yoy else None,
                                'ytd_yoy': float(ytd_yoy) if ytd_yoy else None
                            }

                            all_data.append(data_dict)

                    except (ValueError, IndexError) as e:
                        print(f"연도 {year}, 월 {month_text} 데이터 파싱 오류: {e}")
                        continue

            time.sleep(0.5)

        if not all_data:
            print("크롤링된 데이터가 없습니다.")
            return pd.DataFrame()

        df = pd.DataFrame(all_data)

        # 날짜 순으로 정렬
        df = df.sort_values(['year', 'month']).reset_index(drop=True)

        print(f"\n총 {len(df)}개의 데이터를 수집했습니다.")
        print(f"기간: {df['date'].min()} ~ {df['date'].max()}")

        return df

    except requests.exceptions.RequestException as e:
        print(f"데이터 요청 중 오류 발생: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"예상치 못한 오류 발생: {e}")
        return pd.DataFrame()


def save_to_excel(df, filename='quanta_monthly_sales.xlsx'):
    """
    데이터를 Excel 파일로 저장
    """
    if df.empty:
        print("저장할 데이터가 없습니다.")
        return

    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df.to_excel(writer, sheet_name='Monthly Sales', index=False)

            # 워크시트 가져오기
            worksheet = writer.sheets['Monthly Sales']

            # 컬럼 너비 자동 조정
            for column in worksheet.columns:
                max_length = 0
                column_letter = column[0].column_letter

                for cell in column:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass

                adjusted_width = min(max_length + 2, 50)
                worksheet.column_dimensions[column_letter].width = adjusted_width

        print(f"\n데이터가 '{filename}' 파일로 저장되었습니다.")

    except Exception as e:
        print(f"Excel 저장 중 오류 발생: {e}")


# def main():
#     """
#     메인 실행 함수
#     """
#     print("광달(Quanta Computer) 월간 매출 데이터 크롤링 시작...")
#     print("=" * 60)
#
#     # 데이터 크롤링 (2006년부터 2026년까지)
#     df = crawl_quanta_monthly_sales(start_year=2006, end_year=2026)
#
#     if not df.empty:
#         # 데이터 미리보기
#         print("\n" + "=" * 60)
#         print("데이터 미리보기 (최근 10개):")
#         print(df.tail(10).to_string())
#
#         # 기본 통계
#         print("\n" + "=" * 60)
#         print("기본 통계:")
#         print(df[['current_sales', 'mom', 'yoy', 'ytd_yoy']].describe())
#
#         # Excel 저장
#         save_to_excel(df, 'quanta_monthly_sales.xlsx')
#
#         return df
#     else:
#         print("크롤링 실패")
#         return None
#
#
# if __name__ == "__main__":
#     df = main()

In [2]:
 df = crawl_quanta_monthly_sales(start_year=2006, end_year=2026)

연도 2012, 월 August 데이터 파싱 오류: could not convert string to float: '-6.0,'
연도 2013, 월 January 데이터 파싱 오류: could not convert string to float: '-'

총 239개의 데이터를 수집했습니다.
기간: 2006-01 ~ 2026-01


In [3]:
df

,year,month,date,current_year,current_sales,prev_year,prev_sales,mom,yoy,ytd_yoy
0,2006,1,2006-01,2006,33004.0,2005,25110.0,31.4,31.4,-12.8
1,2006,2,2006-02,2006,33042.0,2005,24534.0,34.7,33.0,0.1
2,2006,3,2006-03,2006,38934.0,2005,29277.0,33.0,33.0,17.8
3,2006,4,2006-04,2006,36030.0,2005,29438.0,22.4,30.1,-7.5
4,2006,5,2006-05,2006,26222.0,2005,28031.0,-6.5,22.6,-27.2
...,...,...,...,...,...,...,...,...,...,...
234,2025,9,2025-09,2025Adjusted Consolidated,184109.0,2024Adjusted Consolidated,155110.0,20.5,18.7,49.5
235,2025,10,2025-10,2025Adjusted Consolidated,173196.0,2024Adjusted Consolidated,135893.0,-5.9,27.4,46.8
236,2025,11,2025-11,2025Adjusted Consolidated,192947.0,2024Adjusted Consolidated,141354.0,11.4,36.5,45.7
237,2025,12,2025-12,2025Adjusted Consolidated,272495.0,2024Adjusted Consolidated,140066.0,41.2,94.5,50.5
